In [34]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings("ignore")

env_path = os.path.abspath(os.path.join(os.getcwd(), "../.env"))
load_dotenv(dotenv_path=env_path)
DB_URL = os.getenv("DB_URL")

engine = create_engine(
    DB_URL,
    pool_size=2,
    max_overflow=0,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 10}
)
print("✅ Ready!")

✅ Ready!


In [35]:
with engine.connect() as conn:
    results    = pd.read_sql("SELECT * FROM results", conn)
    qualifying = pd.read_sql("SELECT * FROM qualifying", conn)
    weather    = pd.read_sql("SELECT * FROM weather", conn)
    laps       = pd.read_sql("SELECT * FROM laps", conn)

print(f"Results:    {results.shape}")
print(f"Qualifying: {qualifying.shape}")
print(f"Weather:    {weather.shape}")
print(f"Laps:       {laps.shape}")

Results:    (589, 12)
Qualifying: (612, 7)
Weather:    (29, 7)
Laps:       (31973, 8)


In [36]:
# avg lap time per driver per race
avg_lap = (
    laps[laps["lap_time_secs"].notna()]
    .groupby(["race_id","driver"])["lap_time_secs"]
    .mean()
    .reset_index()
    .rename(columns={"lap_time_secs": "avg_lap_time"})
)

# driver win rate (historical)
total_races = results.groupby("driver").size().reset_index(name="total_races")
total_wins  = results[results["finish_pos"]==1].groupby("driver").size().reset_index(name="total_wins")
win_rate    = total_races.merge(total_wins, on="driver", how="left").fillna(0)
win_rate["win_rate"] = win_rate["total_wins"] / win_rate["total_races"]

# merge everything
df = results.merge(
    qualifying[["race_id","driver","grid_pos","q1_secs","q2_secs","q3_secs"]],
    on=["race_id","driver"], how="left", suffixes=("","_q")
)
df = df.merge(weather[["race_id","avg_air_temp","avg_track_temp","avg_humidity","rainfall"]], on="race_id", how="left")
df = df.merge(avg_lap, on=["race_id","driver"], how="left")
df = df.merge(win_rate[["driver","win_rate"]], on="driver", how="left")

# target
df["won"] = (df["finish_pos"] == 1).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Total winners: {df['won'].sum()} / {len(df)} rows")
df.head()

Dataset shape: (589, 23)
Total winners: 29 / 589 rows


,race_id,year,round,race_name,circuit,date,driver,team,finish_pos,grid_pos,...,q1_secs,q2_secs,q3_secs,avg_air_temp,avg_track_temp,avg_humidity,rainfall,avg_lap_time,win_rate,won
0,2025_R01,2025,1,Australian Grand Prix,Melbourne,2025-03-16,NOR,McLaren,1.0,1.0,...,75.912,75.415,75.096,15.71,18.94,78.42,1,103.428302,0.241379,1
1,2025_R01,2025,1,Australian Grand Prix,Melbourne,2025-03-16,VER,Red Bull Racing,2.0,3.0,...,76.018,75.565,75.481,15.71,18.94,78.42,1,103.341151,0.275862,0
2,2025_R01,2025,1,Australian Grand Prix,Melbourne,2025-03-16,RUS,Mercedes,3.0,4.0,...,75.971,75.798,75.546,15.71,18.94,78.42,1,103.686340,0.103448,0
3,2025_R01,2025,1,Australian Grand Prix,Melbourne,2025-03-16,ANT,Mercedes,4.0,16.0,...,76.525,NaN,NaN,15.71,18.94,78.42,1,104.579370,0.137931,0
4,2025_R01,2025,1,Australian Grand Prix,Melbourne,2025-03-16,ALB,Williams,5.0,6.0,...,76.245,76.017,75.737,15.71,18.94,78.42,1,104.672389,0.000000,0


In [37]:
le_driver  = LabelEncoder()
le_team    = LabelEncoder()
le_circuit = LabelEncoder()

df["driver_enc"]  = le_driver.fit_transform(df["driver"].astype(str))
df["team_enc"]    = le_team.fit_transform(df["team"].astype(str))
df["circuit_enc"] = le_circuit.fit_transform(df["circuit"].astype(str))

FEATURES = [
    "grid_pos",
    "driver_enc",
    "team_enc",
    "circuit_enc",
    "avg_lap_time",
    "avg_track_temp",
    "avg_humidity",
    "rainfall",
    "q3_secs",
    "win_rate",
]

df_model = df[FEATURES + ["won"]].dropna()
X = df_model[FEATURES]
y = df_model["won"]

print(f"Training samples: {X.shape}")
print(f"Win ratio: {y.mean():.3f}")

Training samples: (277, 10)
Win ratio: 0.105


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = xgb.XGBClassifier(
    n_estimators     = 300,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = 19,
    eval_metric      = "logloss",
    random_state     = 42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("✅ Model trained!")

✅ Model trained!


In [39]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {acc*100:.1f}%\n")
print(classification_report(y_test, y_pred, target_names=["Not Win","Win"]))

✅ Accuracy: 85.7%

              precision    recall  f1-score   support

     Not Win       0.94      0.90      0.92        50
         Win       0.38      0.50      0.43         6

    accuracy                           0.86        56
   macro avg       0.66      0.70      0.67        56
weighted avg       0.88      0.86      0.87        56



In [40]:
importance = pd.DataFrame({
    "feature"   : FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

fig = px.bar(
    importance,
    x="importance",
    y="feature",
    orientation="h",
    title="What predicts an F1 race win?",
    color="importance",
    color_continuous_scale="reds"
)
fig.show()

print("\n🏆 Top predictor:", importance.iloc[0]["feature"])


🏆 Top predictor: grid_pos


In [41]:
# show win probabilities for latest race weekend
latest_race = results["race_id"].max()
latest = df[df["race_id"] == latest_race][FEATURES + ["driver","team"]].dropna()

latest["win_probability"] = model.predict_proba(latest[FEATURES])[:,1]
latest = latest.sort_values("win_probability", ascending=False)

print(f"🏁 Win Probabilities for {latest_race}:\n")
for _, row in latest.iterrows():
    bar = "█" * int(row["win_probability"] * 50)
    print(f"  {row['driver']:4s} ({row['team'][:15]:15s}) {row['win_probability']*100:5.1f}%  {bar}")

🏁 Win Probabilities for 2026_R05:

  RUS  (Mercedes       )  99.4%  █████████████████████████████████████████████████
  ANT  (Mercedes       )  96.9%  ████████████████████████████████████████████████
  NOR  (McLaren        )  30.9%  ███████████████
  PIA  (McLaren        )   3.0%  █
  VER  (Red Bull Racing)   1.2%  
  HAD  (Red Bull Racing)   0.1%  
  HAM  (Ferrari        )   0.1%  
  LEC  (Ferrari        )   0.1%  
  COL  (Alpine         )   0.1%  


In [42]:
# Monaco 2026 prediction using real qualifying data
monaco_quali = q2026[q2026["race_id"] == "2026_R06"].copy()

# get latest driver info
latest_driver = df.sort_values("race_id").groupby("driver").last().reset_index()[["driver","team","avg_lap_time","win_rate"]]

# merge
monaco = monaco_quali.merge(latest_driver, on="driver", how="left")

# fix team column name
monaco = monaco.rename(columns={"team_y": "team"})
if "team_x" in monaco.columns:
    monaco = monaco.drop(columns=["team_x"])

# use Monaco average conditions
monaco["avg_track_temp"] = 28.0
monaco["avg_humidity"]   = 65.0
monaco["rainfall"]       = 0

# encode
monaco["driver_enc"]  = monaco["driver"].apply(
    lambda x: le_driver.transform([x])[0] if x in le_driver.classes_ else -1
)
monaco["team_enc"]    = monaco["team"].apply(
    lambda x: le_team.transform([x])[0] if x in le_team.classes_ else -1
)
monaco["circuit_enc"] = le_circuit.transform(["Monaco"])[0] if "Monaco" in le_circuit.classes_ else 0
monaco["q3_secs"]     = monaco["q3_secs"].fillna(monaco["q1_secs"])

# predict
pred = monaco[FEATURES + ["driver","team"]].dropna()
pred = pred.copy()
pred["win_probability"] = model.predict_proba(pred[FEATURES])[:,1]
pred = pred.sort_values("win_probability", ascending=False)

print("🏁 Monaco 2026 — Predicted Win Probabilities:\n")
for _, row in pred.head(10).iterrows():
    bar = "█" * int(row["win_probability"] * 50)
    print(f"  {row['driver']:4s} ({row['team'][:15]:15s}) Grid P{int(row['grid_pos'])}  {row['win_probability']*100:5.1f}%  {bar}")
    

🏁 Monaco 2026 — Predicted Win Probabilities:

  ANT  (Mercedes       ) Grid P1   99.8%  █████████████████████████████████████████████████
  VER  (Red Bull Racing) Grid P2   77.1%  ██████████████████████████████████████
  HAM  (Ferrari        ) Grid P3    1.9%  
  RUS  (Mercedes       ) Grid P6    1.2%  
  PIA  (McLaren        ) Grid P7    0.8%  
  LEC  (Ferrari        ) Grid P4    0.4%  
  NOR  (McLaren        ) Grid P8    0.4%  
  STR  (Aston Martin   ) Grid P22    0.2%  
  HAD  (Red Bull Racing) Grid P5    0.1%  
  SAI  (Williams       ) Grid P12    0.1%  


In [43]:
# resave encoders with circuit key
with open("../model/encoders.pkl", "wb") as f:
    pickle.dump({
        "driver"  : le_driver,
        "team"    : le_team,
        "circuit" : le_circuit,
        "features": FEATURES
    }, f)

print("✅ encoders.pkl resaved with circuit key!")

✅ encoders.pkl resaved with circuit key!


In [44]:
MODEL_DIR = os.path.join("..", "model")
os.makedirs(MODEL_DIR, exist_ok=True)

with open(f"{MODEL_DIR}/model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(f"{MODEL_DIR}/encoders.pkl", "wb") as f:
    pickle.dump({
        "driver" : le_driver,
        "team"   : le_team,
        "circuit": le_circuit,
        "features": FEATURES
    }, f)

print("✅ model.pkl saved!")
print("✅ encoders.pkl saved!")

✅ model.pkl saved!
✅ encoders.pkl saved!
